[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ronniewillaert/SPM-Textbook-Python/blob/main/notebooks/part-I-introduction/ch04/SPM_Ch4_AFM_Imaging_Modes_Python_Exercises.ipynb)

# AFM Imaging Modes: From Contact to High-Speed
## Interactive Companion to Chapter 4 — *Scanning Probe Microscopy*

This notebook accompanies **Chapter 4** of the textbook *Scanning Probe Microscopy*.

After completing this notebook you will be able to:

- Simulate contact-mode AFM in constant-force and constant-height operation.
- Explore cantilever dynamics as a driven damped harmonic oscillator (resonance, Q-factor).
- Model tapping-mode amplitude–distance curves and understand setpoint selection.
- Interpret phase contrast as a probe of energy dissipation and material properties.
- Calculate frequency shifts in non-contact (FM-AFM) mode from force gradients.
- Map AFM imaging modes onto the tip–sample force–distance curve.
- Simulate AFM scanning with feedback and observe common imaging artifacts.
- Quantify tip convolution effects on lateral resolution.
- Model force-curve-based imaging (QI / PeakForce) and nanomechanical mapping.
- Use an interactive decision tool to select the appropriate AFM mode.

The notebook contains **10 interactive exercises**, each with problem statements, runnable code, plots, and interactive sliders.

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ---------- Physical constants ----------
k_B   = 1.380649e-23    # Boltzmann constant        (J/K)
T_ref = 300              # Room temperature          (K)

# ---------- Matplotlib defaults ----------
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.grid'] = True
plt.rcParams['font.size'] = 11

print("Physical constants loaded ✓")

In [ ]:
# If widgets do not render, run this cell once (Colab).
try:
    from ipywidgets import interact, FloatLogSlider, FloatSlider, IntSlider, Dropdown, HTML, fixed, Checkbox
    import ipywidgets as widgets
    from IPython.display import display, Markdown, clear_output
    print("Widgets ready ✓")
except:
    print("ipywidgets not available, installing...")
    import subprocess
    subprocess.check_call(['pip', '-q', 'install', 'ipywidgets'])
    from ipywidgets import interact, FloatLogSlider, FloatSlider, IntSlider, Dropdown, HTML, fixed, Checkbox
    import ipywidgets as widgets
    from IPython.display import display, Markdown, clear_output
    print("Widgets ready ✓")

---
## 1. Contact Mode: Constant-Force vs Constant-Height Imaging

In **contact mode**, the AFM tip remains in continuous mechanical contact with the surface. Two operating strategies exist:

**Constant-force mode:** A feedback loop adjusts the z-piezo to maintain a constant cantilever deflection (and thus a constant force $F = k \cdot \Delta z_{\text{set}}$). The z-piezo displacement records the topography.

**Constant-height mode:** The z-piezo is held at a fixed height while the cantilever deflection varies freely. The deflection signal records the topography — but only for very flat samples with small height variations.

**Key insight:** Constant-force mode faithfully tracks topography but is limited by feedback bandwidth. Constant-height mode is faster (no feedback delay) but works only on nearly flat surfaces — otherwise the tip crashes or loses contact.

### Tasks
1. Simulate scanning across a surface with steps and bumps.
2. Compare constant-force and constant-height outputs.
3. Vary the feedback bandwidth and observe when constant-force mode fails to track.

In [ ]:
def interactive_contact_mode(k_N_per_m=0.5, setpoint_nm=5.0, fb_bandwidth_Hz=500, scan_speed_um_s=5.0):
    '''
    Simulate contact-mode AFM: constant-force vs constant-height.
    '''
    # Surface profile (1D): step + bump + trench
    N = 2000
    x_um = np.linspace(0, 2.0, N)
    surface_nm = np.zeros(N)
    # Step at 0.5 µm
    surface_nm[x_um > 0.5] += 8.0
    # Gaussian bump at 1.0 µm
    surface_nm += 5.0 * np.exp(-((x_um - 1.0) / 0.05)**2)
    # Trench at 1.5 µm
    surface_nm -= 4.0 * np.exp(-((x_um - 1.5) / 0.03)**2)

    dx_um = x_um[1] - x_um[0]
    dx_m = dx_um * 1e-6
    v_m_s = scan_speed_um_s * 1e-6
    dt = dx_m / v_m_s
    tau = 1.0 / (2 * np.pi * fb_bandwidth_Hz)

    # --- Constant-force mode (PI feedback) ---
    z_piezo_nm = np.zeros(N)
    deflection_cf = np.zeros(N)
    z_piezo_nm[0] = surface_nm[0] - setpoint_nm

    for i in range(1, N):
        # Actual deflection if z_piezo stays
        defl = surface_nm[i] - z_piezo_nm[i-1]
        error = defl - setpoint_nm
        # First-order response with bandwidth limit
        z_piezo_nm[i] = z_piezo_nm[i-1] + (dt / tau) * error * 0.5
        deflection_cf[i] = surface_nm[i] - z_piezo_nm[i]

    # --- Constant-height mode ---
    z_fixed = surface_nm[0] - setpoint_nm
    deflection_ch = surface_nm - z_fixed

    # Force
    force_cf_nN = k_N_per_m * deflection_cf
    force_ch_nN = k_N_per_m * deflection_ch

    print(f"  Contact Mode Imaging Simulation")
    print(f"  ─────────────────────────────────")
    print(f"  Spring constant       k = {k_N_per_m:.2f} N/m")
    print(f"  Deflection setpoint     = {setpoint_nm:.1f} nm  →  Force setpoint = {k_N_per_m * setpoint_nm:.2f} nN")
    print(f"  Feedback bandwidth      = {fb_bandwidth_Hz:.0f} Hz")
    print(f"  Scan speed              = {scan_speed_um_s:.1f} µm/s")
    print()
    print(f"  Constant-force:  RMS tracking error = {np.std(deflection_cf - setpoint_nm):.3f} nm")
    print(f"  Constant-height: deflection range   = {np.ptp(deflection_ch):.1f} nm")

    fig, axes = plt.subplots(2, 2, figsize=(13, 8))

    axes[0, 0].plot(x_um, surface_nm, 'k-', lw=1.5, label='True surface')
    axes[0, 0].plot(x_um, z_piezo_nm, 'b-', lw=2, label='Z-piezo (CF mode)')
    axes[0, 0].axhline(z_fixed, color='r', ls='--', lw=1, label='Z-height (CH mode)')
    axes[0, 0].set_ylabel('Height (nm)')
    axes[0, 0].set_title('Surface tracking')
    axes[0, 0].legend(fontsize=8)

    axes[0, 1].plot(x_um, deflection_cf, 'b-', lw=2, label='Constant-force')
    axes[0, 1].plot(x_um, deflection_ch, 'r-', lw=1.5, alpha=0.7, label='Constant-height')
    axes[0, 1].axhline(setpoint_nm, color='gray', ls=':', lw=1, label=f'Setpoint = {setpoint_nm} nm')
    axes[0, 1].set_ylabel('Cantilever deflection (nm)')
    axes[0, 1].set_title('Deflection signals')
    axes[0, 1].legend(fontsize=8)

    axes[1, 0].plot(x_um, z_piezo_nm, 'b-', lw=2, label='Topography (CF)')
    axes[1, 0].plot(x_um, surface_nm, 'k--', lw=1, alpha=0.5, label='True surface')
    axes[1, 0].set_xlabel('Position (µm)')
    axes[1, 0].set_ylabel('Height (nm)')
    axes[1, 0].set_title('Constant-force topography')
    axes[1, 0].legend(fontsize=8)

    axes[1, 1].plot(x_um, force_ch_nN, 'r-', lw=2, label='Force (CH)')
    axes[1, 1].axhline(k_N_per_m * setpoint_nm, color='gray', ls=':', lw=1)
    axes[1, 1].set_xlabel('Position (µm)')
    axes[1, 1].set_ylabel('Force (nN)')
    axes[1, 1].set_title('Constant-height force signal')
    axes[1, 1].legend(fontsize=8)

    plt.tight_layout()
    plt.show()

interact(
    interactive_contact_mode,
    k_N_per_m=FloatLogSlider(value=0.5, base=10, min=-1, max=2, step=0.1, description='k (N/m)'),
    setpoint_nm=FloatSlider(value=5.0, min=1, max=20, step=0.5, description='Setpoint (nm)'),
    fb_bandwidth_Hz=FloatSlider(value=500, min=50, max=5000, step=50, description='BW (Hz)'),
    scan_speed_um_s=FloatSlider(value=5.0, min=0.5, max=50, step=0.5, description='v (µm/s)')
);

---
## 2. Driven Damped Harmonic Oscillator — Cantilever Resonance

In dynamic AFM modes, the cantilever is modeled as a **driven damped harmonic oscillator**. The steady-state amplitude and phase response are:

$$A(\omega) = \frac{F_0 / m}{\sqrt{(\omega_0^2 - \omega^2)^2 + (\omega \omega_0 / Q)^2}}$$

$$\phi(\omega) = \arctan\!\left(\frac{\omega \omega_0 / Q}{\omega_0^2 - \omega^2}\right)$$

where $\omega_0 = 2\pi f_0$ is the natural resonance frequency, $Q$ is the quality factor, and $F_0$ is the drive amplitude.

**Parameters:**
- **f₀** (resonance frequency): 50–300 kHz in air, higher for small cantilevers
- **Q** (quality factor): ~100–500 in air, ~1–10 in liquid

**Key insight:** Higher Q produces a sharper resonance peak (more sensitive to frequency shifts) but slower response time ($\tau \approx Q / \pi f_0$). In liquid, Q drops dramatically, fundamentally changing the detection strategy.

### Tasks
1. Explore how Q-factor changes the resonance peak width and amplitude.
2. Compare amplitude and phase response curves.
3. Understand why the Q-factor is critical for mode selection (air vs liquid).

In [ ]:
def interactive_harmonic_oscillator(f0_kHz=150.0, Q=200, Q_liquid=5):
    f0 = f0_kHz * 1e3
    omega0 = 2 * np.pi * f0

    # Frequency sweep
    f = np.linspace(f0 * 0.8, f0 * 1.2, 2000)
    omega = 2 * np.pi * f

    # Amplitude and phase for given Q (air)
    def amp_phase(omega, omega0, Q):
        denom = np.sqrt((omega0**2 - omega**2)**2 + (omega * omega0 / Q)**2)
        A = 1.0 / denom  # normalized
        A /= A.max()  # normalize to 1 at peak
        phi = np.arctan2(omega * omega0 / Q, omega0**2 - omega**2)
        return A, np.degrees(phi)

    A_air, phi_air = amp_phase(omega, omega0, Q)
    A_liq, phi_liq = amp_phase(omega, omega0, Q_liquid)

    # Bandwidth (FWHM) and response time
    bw_air = f0 / Q
    bw_liq = f0 / Q_liquid
    tau_air = Q / (np.pi * f0)
    tau_liq = Q_liquid / (np.pi * f0)

    print(f"  Driven Damped Harmonic Oscillator")
    print(f"  ─────────────────────────────────")
    print(f"  Resonance frequency   f₀ = {f0_kHz:.1f} kHz")
    print(f"  Q-factor (air)        Q  = {Q}")
    print(f"  Q-factor (liquid)     Q  = {Q_liquid}")
    print()
    print(f"  {'Parameter':<28s}  {'Air':>12s}  {'Liquid':>12s}")
    print(f"  {'─'*28}  {'─'*12}  {'─'*12}")
    print(f"  {'Resonance bandwidth':<28s}  {bw_air:>10.1f} Hz  {bw_liq:>10.1f} Hz")
    print(f"  {'Response time τ':<28s}  {tau_air*1e6:>10.1f} µs  {tau_liq*1e6:>10.1f} µs")
    print(f"  {'Peak sharpness':<28s}  {'Sharp':>12s}  {'Broad':>12s}")

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

    # Amplitude
    axes[0].plot(f / 1e3, A_air, 'b-', lw=2.5, label=f'Air (Q = {Q})')
    axes[0].plot(f / 1e3, A_liq, 'r-', lw=2.5, label=f'Liquid (Q = {Q_liquid})')
    axes[0].set_xlabel('Frequency (kHz)')
    axes[0].set_ylabel('Normalized amplitude')
    axes[0].set_title('Amplitude response')
    axes[0].legend()

    # Phase
    axes[1].plot(f / 1e3, phi_air, 'b-', lw=2.5, label=f'Air (Q = {Q})')
    axes[1].plot(f / 1e3, phi_liq, 'r-', lw=2.5, label=f'Liquid (Q = {Q_liquid})')
    axes[1].set_xlabel('Frequency (kHz)')
    axes[1].set_ylabel('Phase (degrees)')
    axes[1].set_title('Phase response')
    axes[1].legend()

    # Q sweep comparison
    Q_values = [5, 20, 100, 300, 500]
    for Qv in Q_values:
        A_q, _ = amp_phase(omega, omega0, Qv)
        axes[2].plot(f / 1e3, A_q, lw=1.5, label=f'Q = {Qv}')
    axes[2].set_xlabel('Frequency (kHz)')
    axes[2].set_ylabel('Normalized amplitude')
    axes[2].set_title('Effect of Q-factor on peak shape')
    axes[2].legend(fontsize=8)

    plt.tight_layout()
    plt.show()

interact(
    interactive_harmonic_oscillator,
    f0_kHz=FloatSlider(value=150, min=10, max=500, step=5, description='f₀ (kHz)'),
    Q=IntSlider(value=200, min=10, max=1000, step=10, description='Q (air)'),
    Q_liquid=IntSlider(value=5, min=1, max=50, step=1, description='Q (liquid)')
);

---
## 3. Tapping Mode: Amplitude vs Tip–Sample Distance

In **tapping mode (AM-AFM)**, the cantilever oscillates near resonance with free amplitude $A_0$. As the tip approaches the surface, tip–sample interactions damp the oscillation:

$$A(z) \approx \begin{cases} A_0 & \text{if } z > A_0 \text{ (free oscillation)} \\ z & \text{if } z < A_0 \text{ (amplitude limited by surface)} \end{cases}$$

The **setpoint ratio** $r = A_{\text{set}} / A_0$ determines the imaging force:

- **High ratio** ($r \approx 0.8{-}0.9$): gentle imaging, tip barely touches surface
- **Low ratio** ($r \approx 0.3{-}0.5$): aggressive imaging, deeper penetration into repulsive regime

**Key insight:** A higher setpoint ratio corresponds to a larger working distance $z_{\text{set}}$, meaning the tip spends less time in the repulsive regime and applies lower average force. This is visualized in **Figure 4.8** of the textbook.

### Tasks
1. Simulate the amplitude–distance curve.
2. Mark two setpoints (gentle and aggressive) and compare working distances.
3. Explore how free amplitude and damping affect the operating conditions.

In [ ]:
def interactive_tapping_Az(A0_nm=30.0, setpoint_gentle=0.85, setpoint_aggr=0.50, damping_onset_nm=5.0):
    '''
    Simulate amplitude vs tip-sample distance curve for tapping mode.
    '''
    z = np.linspace(0, A0_nm * 1.8, 1000)  # tip-sample distance (nm)

    # Simple model: A(z) transitions from linear to A0
    # Using a smooth transition
    transition = damping_onset_nm
    A = A0_nm * (1 - np.exp(-(z / A0_nm)**2))
    # Better model: linear ramp below A0, flat above
    A = np.minimum(z, A0_nm)
    # Add smooth onset of damping
    damping_factor = 1.0 / (1.0 + np.exp(-(z - A0_nm + transition) / (transition * 0.3)))
    A = A0_nm * damping_factor
    # Ensure A → A0 far from surface and A → z near surface
    A = np.where(z < A0_nm, np.minimum(z * 0.95, A0_nm * damping_factor), A0_nm * damping_factor)
    # Simplified realistic model
    A = np.where(z >= A0_nm + transition, A0_nm,
         np.where(z <= 0, 0,
                  A0_nm * np.tanh(z / (A0_nm * 0.7))))

    # Setpoint lines
    A_gentle = setpoint_gentle * A0_nm
    A_aggr = setpoint_aggr * A0_nm

    # Find working distances
    idx_gentle = np.argmin(np.abs(A - A_gentle))
    idx_aggr = np.argmin(np.abs(A - A_aggr))
    z_gentle = z[idx_gentle]
    z_aggr = z[idx_aggr]

    print(f"  Tapping Mode: Amplitude vs Distance")
    print(f"  ─────────────────────────────────")
    print(f"  Free amplitude        A₀ = {A0_nm:.1f} nm")
    print(f"  Gentle setpoint       r  = {setpoint_gentle:.2f}  →  A_set = {A_gentle:.1f} nm,  z = {z_gentle:.1f} nm")
    print(f"  Aggressive setpoint   r  = {setpoint_aggr:.2f}  →  A_set = {A_aggr:.1f} nm,  z = {z_aggr:.1f} nm")
    print(f"  Working distance difference Δz = {z_gentle - z_aggr:.1f} nm")
    print()
    print(f"  Physical interpretation:")
    print(f"  • Gentle:     tip barely penetrates attractive regime → minimal sample deformation")
    print(f"  • Aggressive: tip reaches repulsive regime → stronger interaction, higher contrast")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

    # A(z) curve with setpoints
    ax1.plot(z, A, 'b-', lw=2.5, label='A(z) curve')
    ax1.axhline(A0_nm, color='gray', ls=':', lw=1, label=f'A₀ = {A0_nm:.0f} nm')
    ax1.axhline(A_gentle, color='green', ls='--', lw=2, label=f'Gentle: {setpoint_gentle:.0%} A₀')
    ax1.axhline(A_aggr, color='red', ls='--', lw=2, label=f'Aggressive: {setpoint_aggr:.0%} A₀')
    ax1.plot(z_gentle, A_gentle, 'go', ms=12, zorder=5)
    ax1.plot(z_aggr, A_aggr, 'ro', ms=12, zorder=5)
    # Working distance annotations
    ax1.annotate('', xy=(z_gentle, A_gentle - 1), xytext=(z_gentle, 0),
                arrowprops=dict(arrowstyle='<->', color='green', lw=1.5))
    ax1.annotate('', xy=(z_aggr, A_aggr - 1), xytext=(z_aggr, 0),
                arrowprops=dict(arrowstyle='<->', color='red', lw=1.5))
    ax1.set_xlabel('Tip–sample distance z (nm)')
    ax1.set_ylabel('Oscillation amplitude A (nm)')
    ax1.set_title('Amplitude–distance curve')
    ax1.legend(fontsize=8)
    ax1.set_xlim(0, A0_nm * 1.5)

    # Compare different A0 values
    A0_values = [10, 20, 40, 60]
    for a0 in A0_values:
        z_plot = np.linspace(0, a0 * 1.8, 500)
        A_plot = a0 * np.tanh(z_plot / (a0 * 0.7))
        ax2.plot(z_plot / a0, A_plot / a0, lw=2, label=f'A₀ = {a0} nm')
    ax2.axhline(setpoint_gentle, color='green', ls='--', lw=1.5, alpha=0.5)
    ax2.axhline(setpoint_aggr, color='red', ls='--', lw=1.5, alpha=0.5)
    ax2.set_xlabel('Normalized distance z / A₀')
    ax2.set_ylabel('Normalized amplitude A / A₀')
    ax2.set_title('Normalized A(z) curves')
    ax2.legend()

    plt.tight_layout()
    plt.show()

interact(
    interactive_tapping_Az,
    A0_nm=FloatSlider(value=30, min=5, max=100, step=1, description='A₀ (nm)'),
    setpoint_gentle=FloatSlider(value=0.85, min=0.5, max=0.99, step=0.01, description='r (gentle)'),
    setpoint_aggr=FloatSlider(value=0.50, min=0.1, max=0.8, step=0.01, description='r (aggr.)'),
    damping_onset_nm=FloatSlider(value=5.0, min=1, max=20, step=0.5, description='Onset (nm)')
);

---
## 4. Phase Imaging: Energy Dissipation and Material Contrast

In tapping mode, the **phase lag** $\phi$ between the drive signal and the cantilever response encodes information about **energy dissipation** at the tip–sample interface.

The energy dissipated per oscillation cycle is:

$$E_{\text{diss}} \propto A \sin(\phi)$$

**Physical origins of dissipation:**
- Viscoelastic deformation (polymers, biological membranes)
- Adhesion hysteresis (bond formation/rupture)
- Capillary forces (ambient conditions)
- Molecular rearrangements

**Key insight:** Phase contrast reveals material properties **independent of topography**. Two regions with identical height can show very different phase signals if they differ in stiffness or adhesion.

### Tasks
1. Simulate a surface with two materials (elastic and viscoelastic).
2. Compute the phase contrast and energy dissipation map.
3. Explore how the drive amplitude and material loss tangent affect phase contrast.

In [ ]:
def interactive_phase_imaging(A0_nm=30.0, phi_elastic_deg=5.0, phi_viscoel_deg=35.0, A_set_ratio=0.7):
    '''
    Simulate phase contrast between elastic and viscoelastic regions.
    '''
    # Create a 1D surface with two materials
    N = 500
    x_um = np.linspace(0, 2.0, N)

    # Topography is flat (both materials at same height)
    topo_nm = np.zeros(N) + 2.0 * np.sin(2 * np.pi * x_um / 2.0)  # gentle undulation

    # Material assignment: elastic (left half), viscoelastic (right half)
    is_viscoel = (x_um > 0.5) & (x_um < 1.0) | (x_um > 1.5)

    # Phase values
    phi_deg = np.where(is_viscoel, phi_viscoel_deg, phi_elastic_deg)
    phi_rad = np.radians(phi_deg)

    # Amplitude (reduced by interaction, approximately the setpoint)
    A_nm = A0_nm * A_set_ratio * np.ones(N)

    # Energy dissipation
    E_diss = A_nm * np.sin(phi_rad)

    # Phase difference
    delta_phi = phi_viscoel_deg - phi_elastic_deg

    print(f"  Phase Imaging: Material Contrast")
    print(f"  ─────────────────────────────────")
    print(f"  Free amplitude     A₀   = {A0_nm:.1f} nm")
    print(f"  Setpoint ratio     r    = {A_set_ratio:.2f}")
    print(f"  Elastic phase      φ₁   = {phi_elastic_deg:.1f}°")
    print(f"  Viscoelastic phase φ₂   = {phi_viscoel_deg:.1f}°")
    print(f"  Phase contrast     Δφ   = {delta_phi:.1f}°")
    print()
    print(f"  E_diss (elastic)        = {A0_nm * A_set_ratio * np.sin(np.radians(phi_elastic_deg)):.2f} (a.u.)")
    print(f"  E_diss (viscoelastic)   = {A0_nm * A_set_ratio * np.sin(np.radians(phi_viscoel_deg)):.2f} (a.u.)")
    print(f"  Dissipation ratio       = {np.sin(np.radians(phi_viscoel_deg)) / np.sin(np.radians(phi_elastic_deg)):.1f}×")

    fig, axes = plt.subplots(2, 2, figsize=(13, 8))

    # Material map
    ax = axes[0, 0]
    colors = np.where(is_viscoel, 1, 0)
    ax.fill_between(x_um, -1, 15, where=is_viscoel, alpha=0.2, color='orange', label='Viscoelastic')
    ax.fill_between(x_um, -1, 15, where=~is_viscoel, alpha=0.2, color='steelblue', label='Elastic')
    ax.plot(x_um, topo_nm, 'k-', lw=2, label='Topography')
    ax.set_ylabel('Height (nm)')
    ax.set_title('Surface: topography + material map')
    ax.legend(fontsize=8)
    ax.set_ylim(-1, 6)

    # Phase signal
    ax = axes[0, 1]
    ax.plot(x_um, phi_deg, 'r-', lw=2.5)
    ax.axhline(phi_elastic_deg, color='steelblue', ls=':', lw=1, label=f'Elastic: {phi_elastic_deg:.0f}°')
    ax.axhline(phi_viscoel_deg, color='orange', ls=':', lw=1, label=f'Viscoelastic: {phi_viscoel_deg:.0f}°')
    ax.set_ylabel('Phase lag (°)')
    ax.set_title('Phase signal — material contrast')
    ax.legend(fontsize=8)

    # Energy dissipation
    ax = axes[1, 0]
    ax.plot(x_um, E_diss, 'm-', lw=2.5)
    ax.set_xlabel('Position (µm)')
    ax.set_ylabel('E_diss (a.u.)')
    ax.set_title('Energy dissipation: E_diss ∝ A·sin(φ)')

    # Parametric: E_diss vs phase for different amplitudes
    ax = axes[1, 1]
    phi_range = np.linspace(0, 90, 200)
    for a0 in [10, 20, 40, 60]:
        a_set = a0 * A_set_ratio
        e = a_set * np.sin(np.radians(phi_range))
        ax.plot(phi_range, e, lw=2, label=f'A₀ = {a0} nm')
    ax.axvline(phi_elastic_deg, color='steelblue', ls=':', lw=1)
    ax.axvline(phi_viscoel_deg, color='orange', ls=':', lw=1)
    ax.set_xlabel('Phase lag (°)')
    ax.set_ylabel('E_diss (a.u.)')
    ax.set_title('Dissipation vs phase for different amplitudes')
    ax.legend(fontsize=8)

    plt.tight_layout()
    plt.show()

interact(
    interactive_phase_imaging,
    A0_nm=FloatSlider(value=30, min=5, max=80, step=1, description='A₀ (nm)'),
    phi_elastic_deg=FloatSlider(value=5, min=1, max=30, step=1, description='φ elastic (°)'),
    phi_viscoel_deg=FloatSlider(value=35, min=10, max=80, step=1, description='φ viscoel. (°)'),
    A_set_ratio=FloatSlider(value=0.7, min=0.3, max=0.95, step=0.05, description='A_set / A₀')
);

---
## 5. Non-Contact Mode: Frequency Shift and Force Gradient (FM-AFM)

In **FM-AFM (frequency modulation)**, the cantilever oscillates at its resonance frequency, and tip–sample interactions modify the effective spring constant:

$$k_{\text{eff}} = k - \frac{\partial F_{\text{ts}}}{\partial z}$$

This change in stiffness shifts the resonance frequency:

$$\Delta f \approx -\frac{f_0}{2k} \frac{\partial F_{\text{ts}}}{\partial z}$$

**Key insight:** FM-AFM measures the **force gradient** $\partial F / \partial z$, not the force directly. This is why FM-AFM achieves atomic resolution: even when the total force is small, the force gradient can be very large at short range, producing strong contrast.

### Tasks
1. Compute the frequency shift for a Lennard-Jones-type force gradient.
2. Explore how cantilever stiffness and resonance frequency affect sensitivity.
3. Understand the transition from attractive to repulsive force gradients.

In [ ]:
def interactive_fm_afm(f0_kHz=300, k_N_per_m=40.0, E_LJ_eV=0.5, sigma_nm=0.3):
    '''
    Simulate FM-AFM frequency shift from a Lennard-Jones interaction.
    '''
    f0 = f0_kHz * 1e3
    E_LJ = E_LJ_eV * 1.602e-19  # J
    sigma = sigma_nm * 1e-9       # m

    # Distance range
    z_nm = np.linspace(0.25, 3.0, 1000)
    z_m = z_nm * 1e-9

    # Lennard-Jones potential: V(z) = E_LJ * [(sigma/z)^12 - 2*(sigma/z)^6]
    r = sigma / z_m
    V = E_LJ * (r**12 - 2 * r**6)
    F = -np.gradient(V, z_m)       # Force = -dV/dz
    dFdz = np.gradient(F, z_m)     # Force gradient

    # Frequency shift
    delta_f = -f0 / (2 * k_N_per_m) * dFdz

    # Equilibrium distance (V minimum)
    z_eq_idx = np.argmin(V)
    z_eq = z_nm[z_eq_idx]

    # Find where F = 0
    F_sign_change = np.where(np.diff(np.sign(F)))[0]
    z_F0 = z_nm[F_sign_change[0]] if len(F_sign_change) > 0 else z_eq

    print(f"  FM-AFM: Frequency Shift from Force Gradient")
    print(f"  ─────────────────────────────────")
    print(f"  Resonance frequency  f₀  = {f0_kHz:.0f} kHz")
    print(f"  Spring constant      k   = {k_N_per_m:.1f} N/m")
    print(f"  LJ well depth        E   = {E_LJ_eV:.2f} eV")
    print(f"  LJ sigma             σ   = {sigma_nm:.2f} nm")
    print(f"  Equilibrium distance z₀  = {z_eq:.2f} nm")
    print()
    print(f"  Sensitivity factor: Δf/gradient = f₀/(2k) = {f0/(2*k_N_per_m):.1f} Hz/(N/m)")
    print(f"  Max attractive Δf  ≈ {np.min(delta_f[z_nm > 0.3]):.0f} Hz")
    print(f"  Max repulsive Δf   ≈ {np.max(delta_f[z_nm > 0.3]):.0f} Hz")

    fig, axes = plt.subplots(2, 2, figsize=(13, 8))

    # Potential
    axes[0, 0].plot(z_nm, V / 1.602e-19, 'k-', lw=2)
    axes[0, 0].axhline(0, color='gray', lw=0.5)
    axes[0, 0].axvline(z_eq, color='green', ls=':', lw=1, label=f'z₀ = {z_eq:.2f} nm')
    axes[0, 0].set_ylabel('Potential V (eV)')
    axes[0, 0].set_title('Lennard-Jones potential')
    axes[0, 0].set_ylim(-1.5 * E_LJ_eV, 2 * E_LJ_eV)
    axes[0, 0].legend()

    # Force
    axes[0, 1].plot(z_nm, F * 1e9, 'b-', lw=2)
    axes[0, 1].axhline(0, color='gray', lw=0.5)
    axes[0, 1].fill_between(z_nm, F * 1e9, 0, where=F * 1e9 < 0, alpha=0.15, color='blue', label='Attractive')
    axes[0, 1].fill_between(z_nm, F * 1e9, 0, where=F * 1e9 > 0, alpha=0.15, color='red', label='Repulsive')
    axes[0, 1].set_ylabel('Force F (nN)')
    axes[0, 1].set_title('Tip–sample force')
    axes[0, 1].legend()

    # Force gradient
    axes[1, 0].plot(z_nm, dFdz, 'r-', lw=2)
    axes[1, 0].axhline(0, color='gray', lw=0.5)
    axes[1, 0].set_xlabel('Tip–sample distance z (nm)')
    axes[1, 0].set_ylabel('∂F/∂z (N/m)')
    axes[1, 0].set_title('Force gradient (measured by FM-AFM)')
    axes[1, 0].set_ylim(np.percentile(dFdz[z_nm > 0.3], 1) * 1.2,
                        np.percentile(dFdz[z_nm > 0.3], 99) * 1.2)

    # Frequency shift
    axes[1, 1].plot(z_nm, delta_f, 'm-', lw=2.5)
    axes[1, 1].axhline(0, color='gray', lw=0.5)
    axes[1, 1].set_xlabel('Tip–sample distance z (nm)')
    axes[1, 1].set_ylabel('Δf (Hz)')
    axes[1, 1].set_title('FM-AFM frequency shift')
    axes[1, 1].set_ylim(np.percentile(delta_f[z_nm > 0.3], 1) * 1.2,
                        np.percentile(delta_f[z_nm > 0.3], 99) * 1.2)

    plt.tight_layout()
    plt.show()

interact(
    interactive_fm_afm,
    f0_kHz=FloatSlider(value=300, min=50, max=1000, step=10, description='f₀ (kHz)'),
    k_N_per_m=FloatLogSlider(value=40, base=10, min=0, max=2, step=0.1, description='k (N/m)'),
    E_LJ_eV=FloatSlider(value=0.5, min=0.05, max=2.0, step=0.05, description='E (eV)'),
    sigma_nm=FloatSlider(value=0.3, min=0.15, max=0.8, step=0.01, description='σ (nm)')
);

---
## 6. AFM Imaging Modes Mapped onto the Force–Distance Curve

The three principal AFM imaging modes — **contact**, **tapping**, and **non-contact** — correspond to distinct regions of the tip–sample force–distance curve:

- **Contact mode**: operates in the **repulsive regime** (direct mechanical contact)
- **Tapping mode**: spans **both attractive and repulsive regimes** (intermittent contact)
- **Non-contact mode**: operates in the **attractive regime** only (no mechanical contact)

This mapping onto the force–distance curve is the key to understanding **why each mode behaves differently** and measures different physical quantities.

**Key insight:** The choice of AFM imaging mode is fundamentally a choice of which region of the interaction potential to probe — and therefore which physical observable is measured.

### Tasks
1. Visualize the Lennard-Jones force curve with interaction regimes.
2. Mark the operating regions of contact, tapping, and non-contact modes.
3. Explore how the interaction parameters change the boundaries between regimes.

In [ ]:
def interactive_modes_on_fd(E_eV=0.5, sigma_nm=0.35, A_tapping_nm=20.0, z_nc_nm=0.8):
    '''
    Map AFM imaging modes onto the force-distance curve.
    '''
    sigma = sigma_nm * 1e-9
    E = E_eV * 1.602e-19

    z_nm = np.linspace(0.2, 3.0, 2000)
    z_m = z_nm * 1e-9

    # Lennard-Jones force
    r = sigma / z_m
    F_N = 12 * E / sigma * (r**13 - r**7)
    F_nN = F_N * 1e9

    # Equilibrium point (F = 0)
    z_eq_idx = np.argmin(np.abs(F_nN[z_nm > 0.25]))
    z_eq = z_nm[z_nm > 0.25][z_eq_idx]

    # Contact mode: operates at z < z_eq (repulsive)
    z_contact = z_eq * 0.85

    # Non-contact: operates at z = z_nc_nm (attractive, no contact)
    z_nc = z_nc_nm

    # Tapping mode: oscillates with amplitude A around a mean position
    z_tapping_center = z_eq + A_tapping_nm * 0.3 * 1e-9 / sigma  # slightly above equilibrium
    z_tap_min = max(0.25, z_eq - A_tapping_nm * 0.3)
    z_tap_max = z_eq + A_tapping_nm * 0.7

    print(f"  AFM Modes on the Force–Distance Curve")
    print(f"  ─────────────────────────────────")
    print(f"  LJ parameters: E = {E_eV:.2f} eV,  σ = {sigma_nm:.2f} nm")
    print(f"  Equilibrium distance (F = 0): z_eq ≈ {z_eq:.2f} nm")
    print()
    print(f"  {'Mode':<20s}  {'Region':<25s}  {'Measured quantity':<25s}")
    print(f"  {'─'*20}  {'─'*25}  {'─'*25}")
    print(f"  {'Contact':<20s}  {'Repulsive (z < z_eq)':<25s}  {'Force (deflection)':<25s}")
    print(f"  {'Tapping':<20s}  {'Mixed (attractive+repul.)':<25s}  {'Amplitude, Phase':<25s}")
    print(f"  {'Non-contact':<20s}  {'Attractive (z > z_eq)':<25s}  {'Force gradient (Δf)':<25s}")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

    # Force-distance curve with shaded regions
    ax1.plot(z_nm, F_nN, 'k-', lw=2.5)
    ax1.axhline(0, color='gray', lw=0.5)

    # Shade regimes
    ax1.fill_between(z_nm, F_nN, 0, where=(F_nN > 0) & (z_nm < z_eq + 0.1),
                     alpha=0.15, color='red', label='Repulsive regime')
    ax1.fill_between(z_nm, F_nN, 0, where=(F_nN < 0),
                     alpha=0.15, color='blue', label='Attractive regime')

    # Contact mode marker
    F_at_contact = np.interp(z_contact, z_nm, F_nN)
    ax1.plot(z_contact, F_at_contact, 'rs', ms=14, zorder=5, label=f'Contact (z ≈ {z_contact:.2f} nm)')

    # Tapping mode range
    ax1.axvspan(z_tap_min, z_tap_max, alpha=0.2, color='green', label=f'Tapping (A ≈ {A_tapping_nm:.0f} nm)')

    # Non-contact mode marker
    F_at_nc = np.interp(z_nc, z_nm, F_nN)
    ax1.plot(z_nc, F_at_nc, 'b^', ms=14, zorder=5, label=f'Non-contact (z ≈ {z_nc:.2f} nm)')

    ax1.set_xlabel('Tip–sample distance z (nm)', fontsize=12)
    ax1.set_ylabel('Force F (nN)', fontsize=12)
    ax1.set_title('AFM modes on the force–distance curve', fontsize=13)
    ax1.legend(fontsize=8, loc='upper right')
    ax1.set_ylim(min(F_nN) * 1.3, max(max(F_nN[z_nm > 0.25]) * 0.5, 2))

    # Summary comparison
    modes = ['Contact', 'Tapping', 'Non-contact']
    regimes = ['Repulsive', 'Mixed', 'Attractive']
    observables = ['Deflection → F', 'Amplitude, Phase', 'Δf → ∂F/∂z']
    colors_bar = ['#e74c3c', '#27ae60', '#2980b9']
    invasiveness = [3, 1.5, 0.5]
    resolution = [2, 2.5, 3]

    x_bar = np.arange(3)
    width = 0.35
    ax2.bar(x_bar - width/2, invasiveness, width, color=colors_bar, alpha=0.7, label='Invasiveness')
    ax2.bar(x_bar + width/2, resolution, width, color=colors_bar, alpha=0.4, label='Resolution potential')
    ax2.set_xticks(x_bar)
    ax2.set_xticklabels(modes)
    ax2.set_ylabel('Relative score')
    ax2.set_title('Mode comparison: invasiveness vs resolution')
    ax2.legend()

    plt.tight_layout()
    plt.show()

interact(
    interactive_modes_on_fd,
    E_eV=FloatSlider(value=0.5, min=0.1, max=2.0, step=0.05, description='E (eV)'),
    sigma_nm=FloatSlider(value=0.35, min=0.15, max=0.8, step=0.01, description='σ (nm)'),
    A_tapping_nm=FloatSlider(value=20, min=5, max=60, step=1, description='A_tap (nm)'),
    z_nc_nm=FloatSlider(value=0.8, min=0.4, max=2.0, step=0.05, description='z_nc (nm)')
);

---
## 7. AFM Scanning Simulation: Feedback Bandwidth and Artifacts

During AFM imaging, a **feedback loop** continuously adjusts the z-piezo to maintain a constant tip–sample interaction. The finite bandwidth of this loop acts as a **low-pass filter**, limiting the ability to track rapid surface variations.

Common feedback-related artifacts include:
- **Edge rounding**: sharp features appear smoothed
- **Scan-direction asymmetry**: forward and backward scans differ
- **Overshoot/ringing**: oscillations after step edges (if gains too high)

**Key insight:** Increasing scan speed without increasing feedback bandwidth causes progressive degradation of image quality. The critical parameter is the ratio of feature frequency ($f = v/\lambda$) to feedback bandwidth.

### Tasks
1. Simulate scanning across a surface with different features.
2. Compare forward and backward scan lines.
3. Identify when feedback bandwidth becomes insufficient.

In [ ]:
def interactive_scan_artifacts(fb_bw_Hz=500, scan_speed_um_s=5.0, Kp=0.8, surface_type='Step + bump'):
    '''
    Simulate 1D AFM scanning with feedback and observe artifacts.
    '''
    N = 2000
    x_um = np.linspace(0, 2.0, N)

    # Generate surface
    if surface_type == 'Step + bump':
        surface = np.zeros(N)
        surface[x_um > 0.5] += 10.0
        surface += 4.0 * np.exp(-((x_um - 1.2) / 0.04)**2)
        surface -= 3.0 * np.exp(-((x_um - 1.6) / 0.03)**2)
    elif surface_type == 'Fine grating':
        surface = 5.0 * np.sin(2 * np.pi * x_um / 0.1)
    elif surface_type == 'Multi-scale':
        surface = 8.0 * (x_um > 0.5).astype(float) + 3.0 * np.sin(2 * np.pi * x_um / 0.15)
    else:
        surface = 5.0 * np.random.randn(N) * 0.3
        from scipy.ndimage import gaussian_filter1d
        surface = gaussian_filter1d(surface, sigma=10)

    dx_m = (x_um[1] - x_um[0]) * 1e-6
    v_m_s = scan_speed_um_s * 1e-6
    dt = dx_m / v_m_s
    tau = 1.0 / (2 * np.pi * fb_bw_Hz)

    # Forward scan
    z_fwd = np.zeros(N)
    for i in range(1, N):
        error = surface[i] - z_fwd[i-1]
        z_fwd[i] = z_fwd[i-1] + Kp * (dt / tau) * error

    # Backward scan
    z_bwd = np.zeros(N)
    z_bwd[-1] = z_fwd[-1]
    for i in range(N-2, -1, -1):
        error = surface[i] - z_bwd[i+1]
        z_bwd[i] = z_bwd[i+1] + Kp * (dt / tau) * error

    err_fwd = surface - z_fwd
    err_bwd = surface - z_bwd

    print(f"  AFM Scanning Artifact Simulation")
    print(f"  ─────────────────────────────────")
    print(f"  Feedback bandwidth  = {fb_bw_Hz:.0f} Hz")
    print(f"  Scan speed          = {scan_speed_um_s:.1f} µm/s")
    print(f"  Surface type        = {surface_type}")
    print(f"  RMS error (forward) = {np.std(err_fwd):.3f} nm")
    print(f"  RMS error (backward)= {np.std(err_bwd):.3f} nm")
    print(f"  Fwd/Bwd asymmetry   = {np.std(z_fwd - z_bwd):.3f} nm")

    fig, axes = plt.subplots(2, 2, figsize=(13, 8))

    axes[0, 0].plot(x_um, surface, 'k-', lw=1.5, label='True surface')
    axes[0, 0].plot(x_um, z_fwd, 'b-', lw=2, label='Forward scan')
    axes[0, 0].set_ylabel('Height (nm)')
    axes[0, 0].set_title('Forward scan tracking')
    axes[0, 0].legend(fontsize=8)

    axes[0, 1].plot(x_um, surface, 'k-', lw=1.5, label='True surface')
    axes[0, 1].plot(x_um, z_bwd, 'r-', lw=2, label='Backward scan')
    axes[0, 1].set_ylabel('Height (nm)')
    axes[0, 1].set_title('Backward scan tracking')
    axes[0, 1].legend(fontsize=8)

    axes[1, 0].plot(x_um, z_fwd, 'b-', lw=2, label='Forward')
    axes[1, 0].plot(x_um, z_bwd, 'r-', lw=2, alpha=0.7, label='Backward')
    axes[1, 0].set_xlabel('Position (µm)')
    axes[1, 0].set_ylabel('Height (nm)')
    axes[1, 0].set_title('Forward vs backward — scan direction artifact')
    axes[1, 0].legend(fontsize=8)

    axes[1, 1].plot(x_um, err_fwd, 'b-', lw=1.5, label='Error (fwd)')
    axes[1, 1].plot(x_um, err_bwd, 'r-', lw=1.5, alpha=0.7, label='Error (bwd)')
    axes[1, 1].axhline(0, color='gray', lw=0.5)
    axes[1, 1].set_xlabel('Position (µm)')
    axes[1, 1].set_ylabel('Tracking error (nm)')
    axes[1, 1].set_title('Error signal (diagnostic for artifacts)')
    axes[1, 1].legend(fontsize=8)

    plt.tight_layout()
    plt.show()

interact(
    interactive_scan_artifacts,
    fb_bw_Hz=FloatSlider(value=500, min=50, max=5000, step=50, description='BW (Hz)'),
    scan_speed_um_s=FloatSlider(value=5.0, min=0.5, max=50, step=0.5, description='v (µm/s)'),
    Kp=FloatSlider(value=0.8, min=0.1, max=3.0, step=0.1, description='Kp gain'),
    surface_type=Dropdown(options=['Step + bump', 'Fine grating', 'Multi-scale'],
                          value='Step + bump', description='Surface')
);

---
## 8. Tip Convolution: Geometric Artifacts from Finite Tip Size

The AFM tip has a finite radius $R$. The measured image is a **convolution of the tip geometry and the true surface topography**, not a direct representation of the surface.

For a feature of true width $w$ and height $h$, the measured width is approximately:

$$w_{\text{measured}} \approx w + 2\sqrt{2Rh}$$

This broadening effect is most significant for:
- **Narrow features** (comparable to tip radius)
- **Tall features** (larger broadening)
- **Blunt tips** (larger $R$)

**Key insight:** AFM **overestimates lateral dimensions** while height measurements remain accurate. This is why sharp tips are essential for high lateral resolution, and why "tip deconvolution" is sometimes necessary.

### Tasks
1. Simulate tip scanning over features of different widths.
2. Compute the convolution profile and compare with the true profile.
3. Explore how tip radius affects apparent feature width.

In [ ]:
def interactive_tip_convolution(R_tip_nm=10.0, feature_width_nm=30.0, feature_height_nm=10.0, feature_type='Pillar'):
    '''
    Simulate tip convolution artifact for a parabolic tip scanning over a feature.
    '''
    N = 2000
    x_nm = np.linspace(-100, 100, N)
    dx = x_nm[1] - x_nm[0]

    # True surface profile
    hw = feature_width_nm / 2
    if feature_type == 'Pillar':
        surface = np.where(np.abs(x_nm) < hw, feature_height_nm, 0.0)
    elif feature_type == 'Trench':
        surface = np.where(np.abs(x_nm) < hw, -feature_height_nm, 0.0)
    elif feature_type == 'Triangle':
        surface = np.where(np.abs(x_nm) < hw,
                          feature_height_nm * (1 - np.abs(x_nm) / hw), 0.0)
    else:
        surface = feature_height_nm * np.exp(-(x_nm / (feature_width_nm * 0.3))**2)

    # Tip profile (parabolic approximation): z_tip(x) = x^2 / (2R)
    # The measured profile is the dilation of the surface by the inverted tip
    # For each scan position x0, z_measured = max over tip contact points

    tip_half_width = int(3 * np.sqrt(2 * R_tip_nm * feature_height_nm) / dx) + 10
    tip_x = np.arange(-tip_half_width, tip_half_width + 1) * dx
    tip_z = tip_x**2 / (2 * R_tip_nm)  # parabolic tip shape (inverted)

    # Dilation: measured(x) = max_over_t [surface(x-t) + tip(t)] - but for AFM it's
    # the highest point the tip touches = envelope
    measured = np.zeros(N)
    for i in range(N):
        max_z = -np.inf
        for j, tx in enumerate(tip_x):
            xi = int(i + j - tip_half_width)
            if 0 <= xi < N:
                # Tip cannot go below surface: measured = max contact point
                z_contact = surface[xi] - tip_z[j]
                if z_contact > max_z:
                    max_z = z_contact
        measured[i] = max_z

    # Correct for baseline
    measured = np.maximum(measured, 0) if feature_type != 'Trench' else measured

    # Analytical broadening
    w_meas_theory = feature_width_nm + 2 * np.sqrt(2 * R_tip_nm * feature_height_nm)
    broadening = w_meas_theory - feature_width_nm

    # Measure actual width from simulated profile
    if feature_type != 'Trench':
        half_max = feature_height_nm / 2
        above = measured > half_max
        if np.any(above):
            w_meas_sim = (np.max(x_nm[above]) - np.min(x_nm[above]))
        else:
            w_meas_sim = 0
    else:
        half_max = -feature_height_nm / 2
        below = measured < half_max
        if np.any(below):
            w_meas_sim = (np.max(x_nm[below]) - np.min(x_nm[below]))
        else:
            w_meas_sim = 0

    print(f"  Tip Convolution Artifact")
    print(f"  ─────────────────────────────────")
    print(f"  Tip radius          R = {R_tip_nm:.1f} nm")
    print(f"  Feature type          = {feature_type}")
    print(f"  True width            = {feature_width_nm:.1f} nm")
    print(f"  True height           = {feature_height_nm:.1f} nm")
    print(f"  Theoretical broadening = 2√(2Rh) = {broadening:.1f} nm")
    print(f"  Theoretical measured w = {w_meas_theory:.1f} nm")
    print(f"  Simulated measured w  ≈ {w_meas_sim:.1f} nm")
    print(f"  Width overestimate    ≈ {(w_meas_sim / feature_width_nm - 1) * 100:.0f}%" if feature_width_nm > 0 else "")

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

    # Profile comparison
    axes[0].plot(x_nm, surface, 'k-', lw=2, label='True surface')
    axes[0].plot(x_nm, measured, 'b-', lw=2.5, label='Measured (tip-convolved)')
    axes[0].fill_between(x_nm, surface, measured, alpha=0.15, color='red', label='Broadening artifact')
    axes[0].set_xlabel('Position (nm)')
    axes[0].set_ylabel('Height (nm)')
    axes[0].set_title(f'Tip convolution (R = {R_tip_nm:.0f} nm)')
    axes[0].legend(fontsize=8)

    # Width vs tip radius
    R_range = np.linspace(1, 50, 200)
    w_vs_R = feature_width_nm + 2 * np.sqrt(2 * R_range * feature_height_nm)
    axes[1].plot(R_range, w_vs_R, 'b-', lw=2.5)
    axes[1].axhline(feature_width_nm, color='k', ls=':', lw=1, label=f'True width = {feature_width_nm:.0f} nm')
    axes[1].axvline(R_tip_nm, color='red', ls='--', lw=1.5, label=f'Current R = {R_tip_nm:.0f} nm')
    axes[1].set_xlabel('Tip radius R (nm)')
    axes[1].set_ylabel('Measured width (nm)')
    axes[1].set_title('Width broadening vs tip radius')
    axes[1].legend(fontsize=8)

    # Width vs feature height
    h_range = np.linspace(0.5, 30, 200)
    w_vs_h = feature_width_nm + 2 * np.sqrt(2 * R_tip_nm * h_range)
    axes[2].plot(h_range, w_vs_h, 'r-', lw=2.5)
    axes[2].axhline(feature_width_nm, color='k', ls=':', lw=1, label=f'True width = {feature_width_nm:.0f} nm')
    axes[2].axvline(feature_height_nm, color='blue', ls='--', lw=1.5, label=f'Current h = {feature_height_nm:.0f} nm')
    axes[2].set_xlabel('Feature height h (nm)')
    axes[2].set_ylabel('Measured width (nm)')
    axes[2].set_title('Width broadening vs feature height')
    axes[2].legend(fontsize=8)

    plt.tight_layout()
    plt.show()

interact(
    interactive_tip_convolution,
    R_tip_nm=FloatSlider(value=10, min=1, max=50, step=1, description='R_tip (nm)'),
    feature_width_nm=FloatSlider(value=30, min=5, max=100, step=1, description='Width (nm)'),
    feature_height_nm=FloatSlider(value=10, min=1, max=30, step=0.5, description='Height (nm)'),
    feature_type=Dropdown(options=['Pillar', 'Trench', 'Triangle', 'Gaussian'],
                          value='Pillar', description='Feature')
);

---
## 9. Force-Curve-Based Imaging: QI Mode and PeakForce Tapping

In **Quantitative Imaging (QI)** and **PeakForce Tapping**, a complete force–distance curve is acquired at every pixel. This enables extraction of multiple nanomechanical parameters:

- **Topography** (height at contact point or setpoint force)
- **Adhesion** (pull-off force during retraction)
- **Stiffness** (slope of the contact region)
- **Deformation** (indentation depth at peak force)
- **Dissipation** (area between approach and retract curves)

The contact region can be modeled with the **Hertz model** for a spherical indenter:

$$F = \frac{4}{3} E^* \sqrt{R} \, \delta^{3/2}$$

where $E^*$ is the reduced modulus, $R$ is the tip radius, and $\delta$ is the indentation.

**Key insight:** QI / PeakForce modes transform each pixel from a single height value into a **complete mechanical characterization**, enabling spatially resolved nanomechanical mapping.

### Tasks
1. Simulate force–distance curves on materials with different stiffness.
2. Extract nanomechanical parameters from the curves.
3. Build a spatially resolved stiffness map from pixel-by-pixel curves.

In [ ]:
def interactive_qi_mode(E_soft_MPa=1.0, E_stiff_MPa=100.0, R_tip_nm=20.0, F_max_nN=5.0, adhesion_nN=0.5):
    '''
    Simulate QI mode: force curves and nanomechanical parameter extraction.
    '''
    R = R_tip_nm * 1e-9
    F_max = F_max_nN * 1e-9

    # Hertz model: F = (4/3) * E* * sqrt(R) * delta^(3/2)
    # Invert: delta = (F / ((4/3) * E* * sqrt(R)))^(2/3)

    delta_max_nm = 20
    delta = np.linspace(0, delta_max_nm, 500) * 1e-9  # m

    E_soft = E_soft_MPa * 1e6   # Pa
    E_stiff = E_stiff_MPa * 1e6 # Pa

    # Force-indentation curves (approach)
    F_soft = (4/3) * E_soft * np.sqrt(R) * delta**1.5
    F_stiff = (4/3) * E_stiff * np.sqrt(R) * delta**1.5

    # Retraction: add adhesion hysteresis
    F_soft_ret = F_soft - adhesion_nN * 1e-9 * np.exp(-delta / (2e-9))
    F_stiff_ret = F_stiff - adhesion_nN * 0.3 * 1e-9 * np.exp(-delta / (1e-9))

    # Find indentation at F_max
    idx_soft = np.argmin(np.abs(F_soft - F_max))
    idx_stiff = np.argmin(np.abs(F_stiff - F_max))
    delta_soft_nm = delta[idx_soft] * 1e9
    delta_stiff_nm = delta[idx_stiff] * 1e9

    # Stiffness (slope at F_max)
    slope_soft = np.gradient(F_soft, delta)[idx_soft]
    slope_stiff = np.gradient(F_stiff, delta)[idx_stiff]

    print(f"  QI Mode: Nanomechanical Parameter Extraction")
    print(f"  ─────────────────────────────────")
    print(f"  Tip radius       R     = {R_tip_nm:.0f} nm")
    print(f"  Peak force       F_max = {F_max_nN:.1f} nN")
    print(f"  Adhesion               = {adhesion_nN:.2f} nN")
    print()
    print(f"  {'Parameter':<22s}  {'Soft ({:.0f} MPa)'.format(E_soft_MPa):>18s}  {'Stiff ({:.0f} MPa)'.format(E_stiff_MPa):>18s}")
    print(f"  {'─'*22}  {'─'*18}  {'─'*18}")
    print(f"  {'Deformation (nm)':<22s}  {delta_soft_nm:>18.2f}  {delta_stiff_nm:>18.2f}")
    print(f"  {'Stiffness (N/m)':<22s}  {slope_soft:>18.3f}  {slope_stiff:>18.3f}")
    print(f"  {'Elastic modulus':<22s}  {E_soft_MPa:>15.1f} MPa  {E_stiff_MPa:>15.1f} MPa")

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # Force-indentation curves
    ax = axes[0]
    ax.plot(delta * 1e9, F_soft * 1e9, 'r-', lw=2.5, label=f'Soft ({E_soft_MPa:.0f} MPa) approach')
    ax.plot(delta * 1e9, F_soft_ret * 1e9, 'r--', lw=1.5, label='Soft retract')
    ax.plot(delta * 1e9, F_stiff * 1e9, 'b-', lw=2.5, label=f'Stiff ({E_stiff_MPa:.0f} MPa) approach')
    ax.plot(delta * 1e9, F_stiff_ret * 1e9, 'b--', lw=1.5, label='Stiff retract')
    ax.axhline(F_max_nN, color='gray', ls=':', lw=1, label=f'F_max = {F_max_nN:.1f} nN')
    ax.plot(delta_soft_nm, F_max_nN, 'ro', ms=10, zorder=5)
    ax.plot(delta_stiff_nm, F_max_nN, 'bo', ms=10, zorder=5)
    ax.set_xlabel('Indentation δ (nm)')
    ax.set_ylabel('Force F (nN)')
    ax.set_title('Force–indentation curves')
    ax.legend(fontsize=7)
    ax.set_xlim(0, delta_max_nm)
    ax.set_ylim(-adhesion_nN * 1.5, F_max_nN * 1.3)

    # Simulated spatial map (1D line scan over two materials)
    nx = 100
    x_px = np.arange(nx)
    is_soft = (x_px > 25) & (x_px < 75)
    E_map = np.where(is_soft, E_soft_MPa, E_stiff_MPa)
    deform_map = np.where(is_soft, delta_soft_nm, delta_stiff_nm)
    stiff_map = np.where(is_soft, slope_soft, slope_stiff)

    ax = axes[1]
    ax.plot(x_px, E_map, 'g-', lw=2.5)
    ax.set_xlabel('Pixel position')
    ax.set_ylabel('Elastic modulus (MPa)')
    ax.set_title('Spatial modulus map')
    ax.set_yscale('log')

    ax = axes[2]
    ax.plot(x_px, deform_map, 'm-', lw=2.5, label='Deformation')
    ax.set_xlabel('Pixel position')
    ax.set_ylabel('Deformation (nm)')
    ax.set_title('Spatial deformation map')
    ax.legend()

    plt.tight_layout()
    plt.show()

interact(
    interactive_qi_mode,
    E_soft_MPa=FloatLogSlider(value=1.0, base=10, min=-1, max=2, step=0.1, description='E soft (MPa)'),
    E_stiff_MPa=FloatLogSlider(value=100.0, base=10, min=0, max=4, step=0.1, description='E stiff (MPa)'),
    R_tip_nm=FloatSlider(value=20, min=5, max=50, step=1, description='R_tip (nm)'),
    F_max_nN=FloatSlider(value=5, min=0.5, max=20, step=0.5, description='F_max (nN)'),
    adhesion_nN=FloatSlider(value=0.5, min=0.0, max=5.0, step=0.1, description='Adhesion (nN)')
);

---
## 10. AFM Mode Selection: Interactive Decision Tool

Selecting the appropriate AFM imaging mode depends on **sample properties**, **environment**, and **measurement objectives**. This interactive tool implements the decision logic from **Figure 4.14** of the textbook.

**Key decision criteria:**
- Sample mechanical properties (soft vs hard)
- Sample stability (loosely vs firmly attached)
- Environment (air, liquid, UHV)
- Target information (topography, material contrast, atomic resolution, dynamics)

**Key insight:** There is no universally optimal mode — each represents a trade-off between force control, spatial resolution, quantitative capability, and imaging speed. Tapping mode serves as a robust default for unknown or delicate samples.

### Tasks
1. Select sample properties and observe the recommended mode.
2. Compare the properties of recommended modes.
3. Understand why certain modes are unsuitable for specific conditions.

In [ ]:
def interactive_mode_selection(
    sample_stiffness='Soft (< 1 GPa)',
    environment='Liquid',
    target_info='Nanomechanical properties',
    sample_adhesion='Low',
    need_atomic_resolution=False
):
    '''
    Interactive AFM mode selection decision tool.
    '''
    # Mode database
    modes = {
        'Contact': {
            'force_control': 2, 'resolution': 7, 'quantitative': 3,
            'speed': 8, 'soft_ok': False, 'liquid_ok': True, 'uhv_ok': True,
            'air_ok': True, 'atomic': False, 'nanomech': False,
            'desc': 'Direct force measurement, continuous contact',
            'color': '#e74c3c'
        },
        'Tapping (AM-AFM)': {
            'force_control': 6, 'resolution': 7, 'quantitative': 4,
            'speed': 7, 'soft_ok': True, 'liquid_ok': True, 'uhv_ok': True,
            'air_ok': True, 'atomic': False, 'nanomech': False,
            'desc': 'Intermittent contact, amplitude/phase detection',
            'color': '#27ae60'
        },
        'Non-contact (FM-AFM)': {
            'force_control': 9, 'resolution': 10, 'quantitative': 5,
            'speed': 4, 'soft_ok': True, 'liquid_ok': False, 'uhv_ok': True,
            'air_ok': False, 'atomic': True, 'nanomech': False,
            'desc': 'Force gradient detection, no contact',
            'color': '#2980b9'
        },
        'QI Mode': {
            'force_control': 9, 'resolution': 6, 'quantitative': 10,
            'speed': 3, 'soft_ok': True, 'liquid_ok': True, 'uhv_ok': False,
            'air_ok': True, 'atomic': False, 'nanomech': True,
            'desc': 'Full F-d curve per pixel, no lateral forces',
            'color': '#8e44ad'
        },
        'PeakForce Tapping': {
            'force_control': 8, 'resolution': 7, 'quantitative': 8,
            'speed': 6, 'soft_ok': True, 'liquid_ok': True, 'uhv_ok': False,
            'air_ok': True, 'atomic': False, 'nanomech': True,
            'desc': 'Force-controlled oscillation, continuous scanning',
            'color': '#e67e22'
        },
        'HS-AFM': {
            'force_control': 5, 'resolution': 5, 'quantitative': 3,
            'speed': 10, 'soft_ok': True, 'liquid_ok': True, 'uhv_ok': False,
            'air_ok': True, 'atomic': False, 'nanomech': False,
            'desc': 'High temporal resolution, real-time imaging',
            'color': '#f39c12'
        },
    }

    # Decision logic
    is_soft = 'Soft' in sample_stiffness
    env_key = environment.lower()

    scores = {}
    for name, props in modes.items():
        score = 0
        reasons = []

        # Environment filter
        if 'liquid' in env_key and not props['liquid_ok']:
            score -= 10
            reasons.append('Not suitable for liquid')
        elif 'uhv' in env_key and not props['uhv_ok']:
            score -= 10
            reasons.append('Not suitable for UHV')
        elif 'air' in env_key.lower() and not props['air_ok']:
            score -= 5
            reasons.append('Difficult in air')
        else:
            score += 2

        # Soft sample filter
        if is_soft and not props['soft_ok']:
            score -= 8
            reasons.append('Too invasive for soft samples')
        elif is_soft and props['soft_ok']:
            score += 3

        # Target info
        if 'Nanomechanical' in target_info and props['nanomech']:
            score += 5
            reasons.append('Quantitative nanomechanics')
        elif 'Topography' in target_info:
            score += props['speed'] * 0.3
        elif 'Atomic' in target_info and props['atomic']:
            score += 5
            reasons.append('Atomic resolution capable')
        elif 'Dynamics' in target_info and name == 'HS-AFM':
            score += 5
            reasons.append('Time-resolved imaging')
        elif 'Material' in target_info:
            score += props['quantitative'] * 0.3

        # Atomic resolution
        if need_atomic_resolution:
            if props['atomic']:
                score += 5
            else:
                score -= 3

        # High adhesion
        if sample_adhesion == 'High':
            if name in ['QI Mode', 'PeakForce Tapping']:
                score += 2
            elif name == 'Contact':
                score -= 3

        scores[name] = (score, reasons)

    # Sort by score
    ranked = sorted(scores.items(), key=lambda x: x[1][0], reverse=True)
    best = ranked[0][0]

    print(f"  AFM Mode Selection Decision Tool")
    print(f"  ─────────────────────────────────")
    print(f"  Sample stiffness : {sample_stiffness}")
    print(f"  Environment      : {environment}")
    print(f"  Target info      : {target_info}")
    print(f"  Sample adhesion  : {sample_adhesion}")
    print(f"  Atomic resolution: {'Required' if need_atomic_resolution else 'Not required'}")
    print()
    print(f"  ★ RECOMMENDED MODE: {best}")
    print(f"    {modes[best]['desc']}")
    print()
    print(f"  {'Rank':<6s}  {'Mode':<22s}  {'Score':>6s}  Notes")
    print(f"  {'─'*6}  {'─'*22}  {'─'*6}  {'─'*30}")
    for i, (name, (sc, reasons)) in enumerate(ranked):
        flag = '★' if i == 0 else ' '
        notes = '; '.join(reasons[:2]) if reasons else '—'
        print(f"  {flag} {i+1:<4d}  {name:<22s}  {sc:>6.1f}  {notes}")

    # Radar chart comparison of top 3
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

    # Bar chart of scores
    names = [r[0] for r in ranked]
    scrs = [r[1][0] for r in ranked]
    colors = [modes[n]['color'] for n in names]
    bars = ax1.barh(names[::-1], scrs[::-1], color=colors[::-1], alpha=0.8)
    ax1.set_xlabel('Suitability score')
    ax1.set_title('Mode ranking for your conditions')
    ax1.axvline(0, color='gray', lw=0.5)

    # Property comparison of top 3
    top3 = [r[0] for r in ranked[:3]]
    categories = ['Force control', 'Resolution', 'Quantitative', 'Speed']
    cat_keys = ['force_control', 'resolution', 'quantitative', 'speed']
    x_cat = np.arange(len(categories))
    width = 0.25
    for i, name in enumerate(top3):
        vals = [modes[name][k] for k in cat_keys]
        ax2.bar(x_cat + i * width, vals, width, label=name,
               color=modes[name]['color'], alpha=0.8)
    ax2.set_xticks(x_cat + width)
    ax2.set_xticklabels(categories)
    ax2.set_ylabel('Score (1–10)')
    ax2.set_title('Property comparison — top 3 modes')
    ax2.legend(fontsize=8)
    ax2.set_ylim(0, 11)

    plt.tight_layout()
    plt.show()

interact(
    interactive_mode_selection,
    sample_stiffness=Dropdown(
        options=['Soft (< 1 GPa)', 'Moderate (1–10 GPa)', 'Hard (> 10 GPa)'],
        value='Soft (< 1 GPa)', description='Stiffness'),
    environment=Dropdown(
        options=['Air', 'Liquid', 'UHV (ultra-high vacuum)'],
        value='Liquid', description='Environment'),
    target_info=Dropdown(
        options=['Topography only', 'Material contrast (phase)', 'Nanomechanical properties',
                 'Atomic resolution', 'Dynamics (time-resolved)'],
        value='Nanomechanical properties', description='Target info'),
    sample_adhesion=Dropdown(
        options=['Low', 'Moderate', 'High'],
        value='Low', description='Adhesion'),
    need_atomic_resolution=Checkbox(value=False, description='Atomic resolution?')
);

---
## Summary: Key Takeaways from Chapter 4

### 1. **Contact Mode** — Force-Based Topography
- Continuous contact imaging: constant-force (feedback) or constant-height modes
- Limited by lateral forces and sample deformation; suitable for hard materials

### 2. **Cantilever Dynamics** — Driven Harmonic Oscillator
- Resonance frequency $f_0$ and quality factor $Q$ define the dynamic response
- Higher Q = sharper peak = better sensitivity but slower response
- Q drops dramatically in liquid, changing the detection strategy

### 3. **Tapping Mode** — Setpoint and Interaction Control
- Amplitude–distance curve links setpoint ratio to imaging force
- High setpoint ratio = gentle; low ratio = aggressive
- Most versatile mode for unknown or delicate samples

### 4. **Phase Imaging** — Energy Dissipation Mapping
- Phase contrast reveals material properties independent of topography
- $E_{\text{diss}} \propto A \sin(\phi)$: quantitative dissipation from measurable signals

### 5. **FM-AFM** — Force Gradient Detection
- Frequency shift $\Delta f \propto \partial F / \partial z$: measures force gradient, not force
- Enables atomic resolution through sensitivity to short-range gradients

### 6. **Modes on the Force Curve** — Regime Selection
- Contact = repulsive; Tapping = mixed; Non-contact = attractive
- Mode choice = choice of interaction regime and physical observable

### 7. **Feedback Artifacts** — Bandwidth Limitations
- Finite feedback bandwidth smooths sharp features and introduces scan-direction asymmetry
- Diagnostic: compare forward/backward scans; vary scan speed

### 8. **Tip Convolution** — Geometric Broadening
- Measured width $\approx w + 2\sqrt{2Rh}$: lateral dimensions are overestimated
- Sharp tips essential for high lateral resolution

### 9. **QI / PeakForce** — Quantitative Nanomechanical Mapping
- Full force–distance curve per pixel: topography + adhesion + stiffness + deformation
- Transforms AFM from imaging tool to nanomechanical measurement platform

### 10. **Mode Selection** — Application-Dependent Trade-Offs
- No single mode optimizes all dimensions simultaneously
- Trade-off between force control, resolution, quantitative capability, and speed

---

## Notebook Complete

You have now explored **all major imaging mode topics from Chapter 4**:

1. Contact mode (constant-force vs constant-height)
2. Cantilever resonance dynamics (harmonic oscillator, Q-factor)
3. Tapping mode amplitude–distance curves and setpoint selection
4. Phase imaging and energy dissipation
5. FM-AFM frequency shift and force gradient detection
6. AFM modes mapped onto the force–distance curve
7. Scanning artifacts from feedback bandwidth limitations
8. Tip convolution and geometric broadening
9. Force-curve-based imaging (QI / PeakForce) and nanomechanical mapping
10. AFM mode selection decision tool

**Next steps:**
- Try the conceptual and numerical problems in Section 4.6 of the textbook
- Compare your simulation results with the figures in Chapter 4
- Explore how mode selection affects the force–distance curves studied in Chapter 5

---

**Notebook authored for:** *Scanning Probe Microscopy* textbook, Chapter 4